# Video Snippet Extraction by Individual

This notebook extracts video clips containing each proposed individual macaque:

1. Load detection results with individual IDs
2. Identify video segments for each individual
3. Extract clips with temporal buffers
4. Create organized video libraries by individual
5. Generate summary videos and statistics


In [ ]:
import sys
import os
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
import cv2
import warnings
warnings.filterwarnings('ignore')

from macaque_tracker.video_utils import VideoProcessor, VideoClipExtractor, find_video_files

plt.style.use('default')
sns.set_palette("husl")

## 1. Load Results from Previous Steps

In [ ]:
# Find available result files
output_dir = Path("../output")

enhanced_detection_files = list(output_dir.glob("detections_with_individuals_*.csv"))
individual_mapping_files = list(output_dir.glob("individual_mapping_*.csv"))
clustering_summary_files = list(output_dir.glob("clustering_summary_*.csv"))

print(f"Found {len(enhanced_detection_files)} enhanced detection files")
print(f"Found {len(individual_mapping_files)} individual mapping files")
print(f"Found {len(clustering_summary_files)} clustering summary files")

if enhanced_detection_files and individual_mapping_files:
    # Use the first available set of results
    detection_file = enhanced_detection_files[0]
    mapping_file = individual_mapping_files[0]
    
    print(f"\nUsing:")
    print(f"Detections: {detection_file.name}")
    print(f"Mapping: {mapping_file.name}")
    
    if clustering_summary_files:
        summary_file = clustering_summary_files[0]
        print(f"Summary: {summary_file.name}")
else:
    print("\n⚠️  No clustering results found. Please run the clustering notebook first.")

In [ ]:
# Load the data
if enhanced_detection_files and individual_mapping_files:
    df_detections = pd.read_csv(detection_file)
    df_mapping = pd.read_csv(mapping_file)
    
    print(f"Loaded {len(df_detections)} detections")
    print(f"Loaded {len(df_mapping)} track-to-individual mappings")
    
    if clustering_summary_files:
        df_summary = pd.read_csv(summary_file)
        display(df_summary)
    
    # Overview of individuals
    individual_overview = df_detections.groupby('individual_id').agg({
        'track_id': 'nunique',
        'frame': ['count', 'min', 'max'],
        'confidence': 'mean'
    }).round(3)
    
    individual_overview.columns = ['num_tracks', 'total_detections', 'first_frame', 'last_frame', 'avg_confidence']
    individual_overview['duration_frames'] = individual_overview['last_frame'] - individual_overview['first_frame']
    
    print("\nIndividual Overview:")
    display(individual_overview)
    
    # Focus on valid individuals (not noise)
    valid_individuals = individual_overview[individual_overview.index >= 0]
    print(f"\nFound {len(valid_individuals)} proposed individuals (excluding noise)")

## 2. Identify Source Video File

In [ ]:
# Determine which video file was processed
data_dir = "../data/raw"
video_files = find_video_files(data_dir)

# Extract video identifier from detection filename
video_identifier = detection_file.stem.replace('detections_with_individuals_', '')
print(f"Looking for video matching identifier: '{video_identifier}'")

# Find matching video file
source_video = None
for video_file in video_files:
    video_name = Path(video_file).stem
    if video_identifier in video_name or video_name in video_identifier:
        source_video = video_file
        break

if source_video:
    print(f"✓ Found source video: {Path(source_video).name}")
    print(f"  Full path: {source_video}")
    
    # Verify video accessibility
    with VideoProcessor(source_video) as processor:
        print(f"  Duration: {processor.duration:.1f}s ({processor.frame_count} frames @ {processor.fps:.1f} fps)")
else:
    print("\n⚠️  Could not find source video file")
    print("Available videos:")
    for video in video_files[:5]:
        print(f"  {Path(video).name}")
    
    # Use first available video as fallback
    if video_files:
        source_video = video_files[0]
        print(f"\nUsing first available video as fallback: {Path(source_video).name}")

## 3. Configure Clip Extraction Parameters

In [ ]:
# Configuration parameters
BUFFER_SECONDS = 3.0  # Seconds to add before/after detection sequences
MIN_CLIP_DURATION = 1.0  # Minimum clip duration in seconds
MAX_CLIP_DURATION = 30.0  # Maximum clip duration in seconds
MAX_CLIPS_PER_INDIVIDUAL = 10  # Limit clips per individual to manage storage

# Create output directory for clips
clips_dir = Path("../output/clips")
clips_dir.mkdir(parents=True, exist_ok=True)

print(f"Clip extraction configuration:")
print(f"  Buffer time: {BUFFER_SECONDS} seconds")
print(f"  Clip duration range: {MIN_CLIP_DURATION}-{MAX_CLIP_DURATION} seconds")
print(f"  Max clips per individual: {MAX_CLIPS_PER_INDIVIDUAL}")
print(f"  Output directory: {clips_dir}")

## 4. Analyze Temporal Patterns for Each Individual

In [ ]:
# Analyze temporal distribution of detections for each individual
if source_video and 'valid_individuals' in locals() and not valid_individuals.empty:
    fig, axes = plt.subplots(len(valid_individuals), 1, 
                            figsize=(12, 2*len(valid_individuals)), 
                            squeeze=False)
    axes = axes.flatten()
    
    individual_segments = {}
    
    with VideoProcessor(source_video) as processor:
        fps = processor.fps
        
        for i, individual_id in enumerate(valid_individuals.index):
            individual_detections = df_detections[df_detections['individual_id'] == individual_id]
            frames = individual_detections['frame'].values
            times = frames / fps  # Convert to seconds
            
            # Plot detection timeline
            axes[i].scatter(times, [individual_id] * len(times), alpha=0.6, s=10)
            axes[i].set_ylabel(f'Individual {individual_id}')
            axes[i].set_xlim(0, processor.duration)
            axes[i].grid(True, alpha=0.3)
            
            # Identify continuous segments
            segments = []
            if len(frames) > 0:
                segment_start = frames[0]
                segment_end = frames[0]
                
                for frame in frames[1:]:
                    if frame <= segment_end + fps * BUFFER_SECONDS * 2:  # Allow gap up to 2x buffer
                        segment_end = frame
                    else:
                        segments.append((segment_start, segment_end))
                        segment_start = frame
                        segment_end = frame
                
                segments.append((segment_start, segment_end))
                
            individual_segments[individual_id] = segments
            
            # Highlight segments
            for start_frame, end_frame in segments:
                start_time = start_frame / fps
                end_time = end_frame / fps
                axes[i].axvspan(start_time, end_time, alpha=0.2, color='red')
            
            axes[i].set_title(f'Individual {individual_id}: {len(segments)} segments, {len(times)} detections')
    
    axes[-1].set_xlabel('Time (seconds)')
    plt.tight_layout()
    plt.show()
    
    print("Temporal Segments Summary:")
    for individual_id, segments in individual_segments.items():
        total_duration = sum([(end-start)/fps for start, end in segments])
        print(f"Individual {individual_id}: {len(segments)} segments, {total_duration:.1f}s total")

## 5. Extract Video Clips

In [ ]:
# Initialize clip extractor
if source_video and 'individual_segments' in locals():
    extractor = VideoClipExtractor(output_dir=str(clips_dir))
    
    # Create individual mapping for extraction
    track_to_individual_dict = dict(zip(df_mapping['track_id'], df_mapping['individual_id']))
    
    print(f"Starting clip extraction for {len(valid_individuals)} individuals...")
    
    try:
        clips_by_individual = extractor.extract_clips_by_individual(
            video_path=source_video,
            detections_df=df_detections,
            individual_mapping=track_to_individual_dict,
            buffer_seconds=BUFFER_SECONDS
        )
        
        print(f"\n✓ Clip extraction completed!")
        
        # Summary of extracted clips
        total_clips = sum(len(clips) for clips in clips_by_individual.values())
        print(f"Total clips extracted: {total_clips}")
        
        for individual_id, clips in clips_by_individual.items():
            print(f"  Individual {individual_id}: {len(clips)} clips")
            for clip_path in clips[:3]:  # Show first 3 clips
                print(f"    - {Path(clip_path).name}")
            if len(clips) > 3:
                print(f"    ... and {len(clips) - 3} more")
    
    except Exception as e:
        print(f"\n❌ Error during clip extraction: {e}")
        clips_by_individual = {}
else:
    print("Cannot extract clips - missing source video or segment data")
    clips_by_individual = {}

## 6. Create Individual Video Libraries

In [ ]:
# Organize clips into individual directories
if clips_by_individual:
    individual_dirs = {}
    
    for individual_id, clips in clips_by_individual.items():
        if individual_id >= 0:  # Skip noise
            # Create individual directory
            individual_dir = clips_dir / f"individual_{individual_id:02d}"
            individual_dir.mkdir(exist_ok=True)
            individual_dirs[individual_id] = individual_dir
            
            # Move clips to individual directory
            moved_clips = []
            for clip_path in clips:
                clip_file = Path(clip_path)
                if clip_file.exists():
                    new_path = individual_dir / clip_file.name
                    if not new_path.exists():
                        clip_file.rename(new_path)
                    moved_clips.append(str(new_path))
            
            clips_by_individual[individual_id] = moved_clips
            
            print(f"✓ Individual {individual_id}: {len(moved_clips)} clips in {individual_dir.name}/")
    
    print(f"\n✓ Created {len(individual_dirs)} individual video libraries")

## 7. Generate Clip Statistics and Previews

In [ ]:
# Analyze extracted clips
if clips_by_individual:
    clip_stats = []
    
    for individual_id, clips in clips_by_individual.items():
        if individual_id >= 0 and clips:  # Skip noise and empty clip lists
            total_duration = 0
            total_size_mb = 0
            
            for clip_path in clips:
                if Path(clip_path).exists():
                    # Get file size
                    file_size = Path(clip_path).stat().st_size / (1024*1024)  # MB
                    total_size_mb += file_size
                    
                    # Get video duration
                    try:
                        with VideoProcessor(clip_path) as processor:
                            total_duration += processor.duration
                    except:
                        pass  # Skip if can't read video
            
            clip_stats.append({
                'individual_id': individual_id,
                'num_clips': len(clips),
                'total_duration_sec': total_duration,
                'avg_duration_sec': total_duration / len(clips) if clips else 0,
                'total_size_mb': total_size_mb,
                'avg_size_mb': total_size_mb / len(clips) if clips else 0
            })
    
    if clip_stats:
        df_clip_stats = pd.DataFrame(clip_stats)
        df_clip_stats = df_clip_stats.round(2)
        
        print("Clip Statistics by Individual:")
        display(df_clip_stats)
        
        # Summary statistics
        print(f"\nOverall Statistics:")
        print(f"Total clips: {df_clip_stats['num_clips'].sum()}")
        print(f"Total video duration: {df_clip_stats['total_duration_sec'].sum():.1f} seconds")
        print(f"Total storage: {df_clip_stats['total_size_mb'].sum():.1f} MB")
        print(f"Average clips per individual: {df_clip_stats['num_clips'].mean():.1f}")
        print(f"Average clip duration: {df_clip_stats['avg_duration_sec'].mean():.1f} seconds")


In [ ]:
# Visualize clip statistics
if 'df_clip_stats' in locals() and not df_clip_stats.empty:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Number of clips per individual
    df_clip_stats.set_index('individual_id')['num_clips'].plot(kind='bar', ax=axes[0,0])
    axes[0,0].set_title('Number of Clips per Individual')
    axes[0,0].set_xlabel('Individual ID')
    axes[0,0].set_ylabel('Number of Clips')
    axes[0,0].tick_params(axis='x', rotation=0)
    
    # Total duration per individual
    df_clip_stats.set_index('individual_id')['total_duration_sec'].plot(kind='bar', ax=axes[0,1])
    axes[0,1].set_title('Total Duration per Individual')
    axes[0,1].set_xlabel('Individual ID')
    axes[0,1].set_ylabel('Duration (seconds)')
    axes[0,1].tick_params(axis='x', rotation=0)
    
    # Average clip duration distribution
    axes[1,0].hist(df_clip_stats['avg_duration_sec'], bins=10, alpha=0.7, edgecolor='black')
    axes[1,0].set_title('Distribution of Average Clip Durations')
    axes[1,0].set_xlabel('Average Duration (seconds)')
    axes[1,0].set_ylabel('Number of Individuals')
    
    # File size vs duration
    axes[1,1].scatter(df_clip_stats['total_duration_sec'], df_clip_stats['total_size_mb'])
    axes[1,1].set_title('File Size vs Total Duration')
    axes[1,1].set_xlabel('Total Duration (seconds)')
    axes[1,1].set_ylabel('Total Size (MB)')
    
    # Add individual ID labels
    for _, row in df_clip_stats.iterrows():
        axes[1,1].annotate(f"ID {int(row['individual_id'])}", 
                          (row['total_duration_sec'], row['total_size_mb']),
                          xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.tight_layout()
    plt.show()

## 8. Create Sample Frames from Each Individual

In [ ]:
# Extract sample frames from each individual's clips
if clips_by_individual:
    sample_frames = {}
    
    for individual_id, clips in clips_by_individual.items():
        if individual_id >= 0 and clips:  # Skip noise
            # Use first available clip
            first_clip = clips[0]
            if Path(first_clip).exists():
                try:
                    with VideoProcessor(first_clip) as processor:
                        # Extract middle frame
                        middle_frame = processor.frame_count // 2
                        ret, frame = processor.read_frame(middle_frame)
                        if ret:
                            sample_frames[individual_id] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                except Exception as e:
                    print(f"Could not extract frame from Individual {individual_id}: {e}")
    
    # Display sample frames
    if sample_frames:
        n_individuals = len(sample_frames)
        cols = min(3, n_individuals)
        rows = (n_individuals + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
        if rows == 1 and cols == 1:
            axes = [axes]
        elif rows == 1 or cols == 1:
            axes = axes.flatten()
        else:
            axes = axes.flatten()
        
        for i, (individual_id, frame) in enumerate(sample_frames.items()):
            if i < len(axes):
                axes[i].imshow(frame)
                axes[i].set_title(f'Individual {individual_id}\n({len(clips_by_individual[individual_id])} clips)')
                axes[i].axis('off')
        
        # Hide unused subplots
        for i in range(len(sample_frames), len(axes)):
            axes[i].axis('off')
        
        plt.suptitle('Sample Frames from Each Proposed Individual', fontsize=16)
        plt.tight_layout()
        plt.show()

## 9. Save Results and Create Index

In [ ]:
# Create comprehensive index of all extracted clips
if clips_by_individual:
    clip_index = []
    
    for individual_id, clips in clips_by_individual.items():
        if individual_id >= 0:  # Skip noise
            for i, clip_path in enumerate(clips):
                clip_file = Path(clip_path)
                if clip_file.exists():
                    try:
                        with VideoProcessor(clip_path) as processor:
                            duration = processor.duration
                            frame_count = processor.frame_count
                            fps = processor.fps
                    except:
                        duration = 0
                        frame_count = 0
                        fps = 0
                    
                    clip_index.append({
                        'individual_id': individual_id,
                        'clip_number': i + 1,
                        'filename': clip_file.name,
                        'full_path': str(clip_file),
                        'directory': clip_file.parent.name,
                        'duration_sec': duration,
                        'frame_count': frame_count,
                        'fps': fps,
                        'file_size_mb': clip_file.stat().st_size / (1024*1024)
                    })
    
    if clip_index:
        df_clip_index = pd.DataFrame(clip_index)
        df_clip_index = df_clip_index.round(3)
        
        # Save clip index
        index_file = output_dir / f"clip_index_{video_identifier}.csv"
        df_clip_index.to_csv(index_file, index=False)
        print(f"✓ Saved clip index to {index_file}")
        
        # Save clip statistics
        if 'df_clip_stats' in locals():
            stats_file = output_dir / f"clip_statistics_{video_identifier}.csv"
            df_clip_stats.to_csv(stats_file, index=False)
            print(f"✓ Saved clip statistics to {stats_file}")
        
        print(f"\n📁 Video clips organized in: {clips_dir}")
        print(f"📄 Total files created: {len(df_clip_index)} video clips")
        print(f"📊 Individual directories: {len([d for d in clips_dir.iterdir() if d.is_dir()])}")
        
        # Show first few entries of the index
        print("\nSample of Clip Index:")
        display(df_clip_index.head(10))
else:
    print("No clips to index")

## 10. Create Usage Instructions

In [ ]:
# Generate usage instructions file
instructions = f"""
# Macaque Individual Video Clips - Usage Instructions

Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
Source video: {Path(source_video).name if source_video else 'Unknown'}
Processing method: {df_summary['method'].iloc[0] if 'df_summary' in locals() and not df_summary.empty else 'Unknown'}

## Directory Structure

```
output/clips/
├── individual_00/    # Clips for Individual 0
├── individual_01/    # Clips for Individual 1
├── individual_02/    # Clips for Individual 2
└── ...
```

## Files Created

- `clip_index_{video_identifier}.csv`: Complete index of all extracted clips
- `clip_statistics_{video_identifier}.csv`: Summary statistics per individual
- Individual directories: `individual_XX/` containing video clips

## Individual Summary

"""

if 'df_clip_stats' in locals() and not df_clip_stats.empty:
    instructions += "\n"
    for _, row in df_clip_stats.iterrows():
        instructions += f"- Individual {int(row['individual_id']):02d}: {int(row['num_clips'])} clips, {row['total_duration_sec']:.1f}s total\n"

instructions += f"""

## Usage Notes

1. Each individual directory contains video clips where that individual was detected
2. Clips include {BUFFER_SECONDS}s buffer before and after detection sequences
3. Clip filenames include the source video name and clip number
4. Use the clip index CSV to programmatically access specific clips
5. Individual IDs are based on clustering analysis - manual validation recommended

## Next Steps

1. Review sample clips for each proposed individual
2. Validate individual identifications manually
3. Merge or split individuals as needed
4. Use clips for behavioral analysis or individual recognition training

"""

# Save instructions
instructions_file = clips_dir / "README.txt"
with open(instructions_file, 'w') as f:
    f.write(instructions)

print(f"✓ Created usage instructions: {instructions_file}")
print("\n" + "="*60)
print(instructions)

## Summary

This notebook successfully extracted video clips for each proposed individual macaque:

### Accomplishments:
- ✅ Loaded individual identification results from clustering analysis
- ✅ Identified temporal segments for each individual
- ✅ Extracted video clips with appropriate temporal buffers
- ✅ Organized clips into individual-specific directories
- ✅ Generated comprehensive statistics and previews
- ✅ Created searchable index of all extracted clips
- ✅ Provided usage instructions and next steps

### Files Created:
- `../output/clips/individual_XX/` - Individual video directories
- `../output/clip_index_*.csv` - Complete clip inventory
- `../output/clip_statistics_*.csv` - Summary statistics per individual
- `../output/clips/README.txt` - Usage instructions

### Research Applications:
1. **Individual Behavior Analysis**: Study behavior patterns specific to each macaque
2. **Social Interaction Studies**: Analyze clips where multiple individuals appear
3. **Individual Recognition**: Train machine learning models for automatic ID
4. **Manual Validation**: Review and refine individual identifications
5. **Temporal Analysis**: Study activity patterns and site usage by individuals

The video library is now ready for detailed behavioral analysis and individual tracking studies!